# 04 — Leakage-controlled features and four-week target

**Purpose:** turn the weekly panel into a supervised learning table with 16 predictors and one four-week aggregate target.

**Input:** `../data/processed/weekly_panel.csv`  
**Output:** `../data/processed/model_table.csv`

At forecast origin week *t*, lagged and rolling sales/order/price inputs use information through week *t-1*. The target is the sum of units in weeks *t*, *t+1*, *t+2* and *t+3*. It is one aggregate number, not four separate weekly forecasts.

The first rows of each product history are expected to have missing lag/rolling values. Rows without all required features or a complete four-week target are excluded from the model table.

In [2]:
import pandas as pd
import numpy as np

panel = pd.read_csv("../data/processed/weekly_panel.csv", parse_dates=["week"],
                    dtype={"StockCode": str})
panel = panel.sort_values(["StockCode", "week"]).reset_index(drop=True)
g = panel.groupby("StockCode")
print(panel.shape)

(407702, 5)


## Predictor definitions

| Feature | Meaning |
|---|---|
| `units_lag1`, `units_lag2`, `units_lag4` | Units sold one, two and four weeks earlier |
| `units_mean4`, `units_std4` | Mean and sample standard deviation of the previous four weeks |
| `units_mean8`, `units_std8` | Mean and sample standard deviation of the previous eight weeks |
| `orders_lag1` | Distinct invoices in the previous week |
| `orders4` | Distinct-invoice counts summed across the previous four weeks |
| `price_lag1` | Average selling price in the previous week |
| `price_change4` | Relative change from the average price five week positions earlier to the previous week's price |
| `trend_ratio` | `(units_mean4 + 1) / (units_mean8 + 1)` |
| `week_of_year`, `month` | Calendar position of the forecast-origin week |
| `weeks_to_xmas` | Whole weeks until the next Christmas date |
| `product_age` | Number of elapsed weekly steps since the product's first panel week |

The production implementation in `src/stockout_radar/features.py` validates unique product-week keys and regular seven-day intervals before generating these same features. `cumcount()` for `product_age` is equivalent to elapsed weeks here because the panel is regular.

In [3]:
# Lag features: sales in earlier weeks
for k in [1, 2, 4]:
    panel[f"units_lag{k}"] = g["units"].shift(k)

panel[["StockCode", "week", "units", "units_lag1", "units_lag2", "units_lag4"]].head(8)

,StockCode,week,units,units_lag1,units_lag2,units_lag4
0,10002,2009-12-07,24.0,NaN,NaN,NaN
1,10002,2009-12-14,37.0,24.0,NaN,NaN
2,10002,2009-12-21,14.0,37.0,24.0,NaN
3,10002,2009-12-28,0.0,14.0,37.0,NaN
4,10002,2010-01-04,15.0,0.0,14.0,24.0
5,10002,2010-01-11,191.0,15.0,0.0,37.0
6,10002,2010-01-18,41.0,191.0,15.0,14.0
7,10002,2010-01-25,44.0,41.0,191.0,0.0


In [4]:
# Rolling features: summaries of previous weeks
for w in [4, 8]:
    panel[f"units_mean{w}"] = g["units"].transform(lambda s: s.shift(1).rolling(w).mean())
    panel[f"units_std{w}"]  = g["units"].transform(lambda s: s.shift(1).rolling(w).std())

panel[["StockCode", "week", "units", "units_mean4", "units_std4"]].head(10)

,StockCode,week,units,units_mean4,units_std4
0,10002,2009-12-07,24.0,NaN,NaN
1,10002,2009-12-14,37.0,NaN,NaN
2,10002,2009-12-21,14.0,NaN,NaN
3,10002,2009-12-28,0.0,NaN,NaN
4,10002,2010-01-04,15.0,18.75,15.649814
5,10002,2010-01-11,191.0,16.50,15.286159
6,10002,2010-01-18,41.0,55.00,90.924877
7,10002,2010-01-25,44.0,61.75,87.815621
8,10002,2010-02-01,16.0,72.75,79.901502
9,10002,2010-02-08,43.0,73.00,79.661785


`shift(1)` is applied before the rolling calculation so the current forecast-origin week's sales are excluded from the historical input window. Without the shift, the feature would include information from the week being predicted and create target leakage.

In [5]:
# Target: this week plus the following three weeks
H = 4
panel["target"] = sum(g["units"].shift(-k) for k in range(0, H))
print(panel[["StockCode", "week", "units", "target"]].head(6))
print(panel["target"].isna().sum(), "rows have no target")

  StockCode       week  units  target
0     10002 2009-12-07   24.0    75.0
1     10002 2009-12-14   37.0    66.0
2     10002 2009-12-21   14.0   220.0
3     10002 2009-12-28    0.0   247.0
4     10002 2010-01-04   15.0   291.0
5     10002 2010-01-11  191.0   292.0
14009 rows have no target


In [6]:
# Order-frequency features from previous weeks
panel["orders_lag1"] = g["orders"].shift(1)
panel["orders4"] = g["orders"].transform(lambda s: s.shift(1).rolling(4).sum())
panel[["StockCode", "week", "orders", "orders_lag1", "orders4"]].head(8)

,StockCode,week,orders,orders_lag1,orders4
0,10002,2009-12-07,4.0,NaN,NaN
1,10002,2009-12-14,3.0,4.0,NaN
2,10002,2009-12-21,3.0,3.0,NaN
3,10002,2009-12-28,0.0,3.0,NaN
4,10002,2010-01-04,2.0,0.0,10.0
5,10002,2010-01-11,4.0,2.0,8.0
6,10002,2010-01-18,5.0,4.0,9.0
7,10002,2010-01-25,6.0,5.0,11.0


In [7]:
# Price features using historical prices
panel["price_lag1"] = g["avg_price"].shift(1)
panel["price_change4"] = panel["price_lag1"] / g["avg_price"].shift(5) - 1
panel[["StockCode", "week", "avg_price", "price_lag1", "price_change4"]].head(8)

,StockCode,week,avg_price,price_lag1,price_change4
0,10002,2009-12-07,1.062500,NaN,NaN
1,10002,2009-12-14,1.133333,1.062500,NaN
2,10002,2009-12-21,1.133333,1.133333,NaN
3,10002,2009-12-28,1.133333,1.133333,NaN
4,10002,2010-01-04,0.850000,1.133333,NaN
5,10002,2010-01-11,0.850000,0.850000,-0.200000
6,10002,2010-01-18,1.012000,0.850000,-0.250000
7,10002,2010-01-25,0.985000,1.012000,-0.107059


In [8]:
# Short-term versus longer-term demand trend
panel["trend_ratio"] = (panel["units_mean4"] + 1) / (panel["units_mean8"] + 1)
panel[["StockCode", "week", "units_mean4", "units_mean8", "trend_ratio"]].head(10)

,StockCode,week,units_mean4,units_mean8,trend_ratio
0,10002,2009-12-07,NaN,NaN,NaN
1,10002,2009-12-14,NaN,NaN,NaN
2,10002,2009-12-21,NaN,NaN,NaN
3,10002,2009-12-28,NaN,NaN,NaN
4,10002,2010-01-04,18.75,NaN,NaN
5,10002,2010-01-11,16.50,NaN,NaN
6,10002,2010-01-18,55.00,NaN,NaN
7,10002,2010-01-25,61.75,NaN,NaN
8,10002,2010-02-01,72.75,45.75,1.577540
9,10002,2010-02-08,73.00,44.75,1.617486


In [9]:
# Calendar and product-age features
panel["week_of_year"] = panel["week"].dt.isocalendar().week.astype(int)
panel["month"] = panel["week"].dt.month

xmas = pd.to_datetime(panel["week"].dt.year.astype(str) + "-12-25")
xmas = xmas.where(panel["week"] <= xmas, xmas + pd.DateOffset(years=1))
panel["weeks_to_xmas"] = (xmas - panel["week"]).dt.days // 7

panel["product_age"] = g.cumcount()
panel[["week", "week_of_year", "month", "weeks_to_xmas"]].drop_duplicates("week").head(8)

,week,week_of_year,month,weeks_to_xmas
0,2009-12-07,50,12,2
1,2009-12-14,51,12,1
2,2009-12-21,52,12,0
3,2009-12-28,53,12,51
4,2010-01-04,1,1,50
5,2010-01-11,2,1,49
6,2010-01-18,3,1,48
7,2010-01-25,4,1,47


In [10]:
# Save the final model table
features = ["units_lag1", "units_lag2", "units_lag4",
            "units_mean4", "units_std4", "units_mean8", "units_std8",
            "orders_lag1", "orders4", "price_lag1", "price_change4",
            "trend_ratio", "week_of_year", "month", "weeks_to_xmas", "product_age"]

model_df = panel.dropna(subset=features + ["target"]).copy()
print("Rows before:", len(panel), "| after dropping NaN:", len(model_df))
print(len(features), "features")
print(model_df["week"].min(), "to", model_df["week"].max())

model_df[["StockCode", "week", "target"] + features].to_csv(
    "../data/processed/model_table.csv", index=False)

Rows before: 407702 | after dropping NaN: 356909
16 features
2010-02-01 00:00:00 to 2011-11-07 00:00:00


## Model-table validation

This verifies the expected schema and uniqueness after removing rows with incomplete feature history or target windows.

In [11]:
expected_features = [
    "units_lag1", "units_lag2", "units_lag4",
    "units_mean4", "units_std4", "units_mean8", "units_std8",
    "orders_lag1", "orders4", "price_lag1", "price_change4",
    "trend_ratio", "week_of_year", "month", "weeks_to_xmas", "product_age"
]
assert features == expected_features, "Feature list differs from the canonical 16-feature schema"
assert not model_df.duplicated(["StockCode", "week"]).any(), "Duplicate model rows found"
assert model_df[features + ["target"]].notna().all().all(), "Missing feature or target value remains"
assert len(model_df) == 356909, f"Expected 356,909 rows for the supplied data, got {len(model_df):,}"

print("Model table validated:", model_df.shape)
print("Feature count:", len(features))
print("Target:", "sum of current and next three weekly unit totals")

Model table validated: (356909, 22)
Feature count: 16
Target: sum of current and next three weekly unit totals
